Parser Agent 

Internal Rich Schema

Rules:

Use null if missing

Do not hallucinate

Confidence reflects completeness + clarity

System Prompt (KEEP SHORT)
You parse Indian addresses.
Extract structured components accurately.
Do not guess missing fields.
Return valid JSON only.


User Prompt Template

Parse this Indian address into structured fields.

Address:
"{RAW_ADDRESS}"

Return JSON exactly in this format:
{
  "components": {
    "house_number": null,
    "street": null,
    "landmark": null,
    "village": null,
    "taluk": null,
    "city": null,
    "district": null,
    "state": null,
    "pincode": null
  },
  "normalized_address": "string",
  "confidence": 0.0
}


STEP 5: Few-Shot Examples (LIMIT TO 3)
Example 1 (Urban)
Input:
"Flat 302, Prestige Sunrise, Whitefield Blr 560066"

Output:
{
  "components": {
    "house_number": "302",
    "street": null,
    "landmark": "Prestige Sunrise",
    "village": null,
    "taluk": "Whitefield",
    "city": "Bengaluru",
    "district": "Bengaluru Urban",
    "state": "Karnataka",
    "pincode": "560066"
  },
  "normalized_address": "Flat 302, Prestige Sunrise, Whitefield, Bengaluru, Karnataka 560066",
  "confidence": 0.92
}

Example 2 (Landmark-based)
Example 3 (Village / Taluk)

STEP 6: JSON Enforcement (MANDATORY)

Reject non-JSON responses

Strip text before/after JSON

Validate all required keys exist

Accuracy < consistency

In [75]:
import  os
from langchain.tools import tool
import json
import re
from dotenv import load_dotenv
load_dotenv()


os.environ["OPENAI_API_KEY"]  = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")



In [ ]:
ADDRESS_SCHEMA = {
    "components": {
        "house_number": None,
        "street": None,
        "landmark": None,
        "area": None,
        "village": None,
        "taluk": None,
        "city": None,
        "district": None,
        "state": None,
        "pincode": None,
    },
    "normalized_address": None,
    "confidence": None,
}

In [ ]:
def normalize_input(raw_address: str) -> str:
    """
    Basic cleanup before passing to LLM.
    """
    if not raw_address or not raw_address.strip():
        raise ValueError("Empty address input")

    address = raw_address.strip()
    address = " ".join(address.split())  # remove extra spaces
    return address


Prompt construction (Code)

In [91]:
def build_parser_prompt(address: str) -> str:
    """
    Creates a strict instruction prompt for the LLM.
    """
    return f"""
You are an address parser.

Return ONLY valid JSON in the following schema:
{ADDRESS_SCHEMA}

Rules:
- If a field is missing, return null
- If abbreviations or short forms are present (e.g., blr, mys, buld),
expand them to standard English forms.
- If a city, state, or landmark is abbreviated, always expand it to its full official form.

- Normalize city and state names to their official names. (e.g., mys -> mysore, blr -> bengaluru)
- If words are concatenated, split them into meaningful components. like DoorNo32 -> Door No 32, SankalpbuildingA302 -> Sankalp building A 302
- Do not hallucinate values
- Do not add extra keys
- Do not include explanation text

Address:
\"\"\"{address}\"\"\"
"""


LLM call (mocked for now)

In [79]:
from langchain_ollama import OllamaLLM
from ollama import chat   # or ollama client you use

# llm = OllamaLLM(
#     # model="qwen2.5:7b-instruct",
#     model = "gemma3:1b",
#     temperature=0.2
# )

# def call_llm(prompt: str) -> str:
#     response = llm.invoke(prompt)
#     # print(response)
#     return response

def call_llm(prompt: str) -> str:
    response = chat(
        model="qwen2.5:7b-instruct",
        messages=[{"role": "user", "content": prompt}]
    )
    return response["message"]["content"]

JSON enforcement & validation (MOST IMPORTANT)

In [80]:
import json

def enforce_json(response: str) -> dict:
    """
    Enforces strict JSON-only output.
    """
    try:
        parsed = json.loads(response)
    except json.JSONDecodeError:
        raise ValueError("LLM output is not valid JSON")

    # Required keys check
    for key in ADDRESS_SCHEMA:
        if key not in parsed:
            raise ValueError(f"Missing key: {key}")

    for component in ADDRESS_SCHEMA["components"]:
        if component not in parsed["components"]:
            raise ValueError(f"Missing component: {component}")

    return parsed


Confidence scoring (simple & explainable)

In [27]:
def normalize_confidence(value) -> float:
    try:
        value = float(value)
        return max(0.0, min(1.0, value))
    except:
        return 0.0

In [66]:
def compute_confidence(components: dict) -> float:
    """
    Confidence based on number of non-null fields.
    """
    # filled = sum(1 for v in components.values() if v)
    # total = len(components)
    # return round(filled / total, 2)

    return normalize_confidence(components.get("confidence", 0.0))



Orchestrator function (ties everything together)

In [89]:
def parse_address(raw_address: str) -> dict:
    address = normalize_input(raw_address)
    prompt = build_parser_prompt(address)

    response = call_llm(prompt)  # later
    parsed = enforce_json(response)
    # parsed["confidence"] = compute_confidence(parsed["components"]
    parsed["confidence"] = compute_confidence(parsed)
    parsed["normalized_address"] = address

    return parsed


In [92]:
response =parse_address("Near govt schl Yelhanka blr 570007")
response

{'components': {'house_number': None,
  'street': None,
  'landmark': 'govt schl',
  'village': None,
  'taluk': None,
  'city': 'bengaluru',
  'district': None,
  'state': None,
  'pincode': '570007'},
 'normalized_address': 'Near govt schl Yelhanka blr 570007',
 'confidence': 0.8}

In [93]:
response_1 = parse_address("Flat302PrestigeSunriseWhitefieldBlr560066")
response_1

{'components': {'house_number': None,
  'street': None,
  'landmark': 'Flat 302',
  'village': None,
  'taluk': None,
  'city': 'bengaluru',
  'district': None,
  'state': None,
  'pincode': '560066'},
 'normalized_address': 'Flat302PrestigeSunriseWhitefieldBlr560066',
 'confidence': 0.8}